# IBKR Option Chain Fetcher

Fetches live option chains from Interactive Brokers (IB Gateway / TWS) via `ib_insync`
and saves them to the local SQLite database used by the rest of the skew framework.

## Prerequisites
- IB Gateway (paper or live) running on `127.0.0.1:4002` (paper) or `127.0.0.1:4001` (live)
- Required packages:
  ```
  pip install ib_insync nest_asyncio
  ```
- In IB Gateway: API → Settings → Enable ActiveX and Socket Clients ✓

## Sections
1. **Connect** – establish IB Gateway connection
2. **Live snapshot** – fetch current chain for one ticker and save to DB
3. **Historical backfill** – pull N trading days of IV history per contract
4. **Disconnect**

In [ ]:
# ── 0. Imports ──────────────────────────────────────────────────────────────
import sys, time, math
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# nest_asyncio lets ib_insync's sync wrappers work inside Jupyter's event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    raise ImportError("Install nest_asyncio:  pip install nest_asyncio")

# Project root on path
ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from skew.data_store import save_option_snapshot, list_snapshots

try:
    from ib_insync import IB, Stock, Option, util
    print("ib_insync OK")
except ImportError:
    raise ImportError("Install ib_insync:  pip install ib_insync")

---
## 1. Configuration

In [ ]:
# ── 1. Config ───────────────────────────────────────────────────────────────
IB_HOST   = "127.0.0.1"
IB_PORT   = 4002          # 4002 = paper gateway, 4001 = live gateway
CLIENT_ID = 10            # must be unique per active IB connection

DB_PATH   = "../data/options.db"

TICKERS   = ["QQQ"]

# Liquidity filters applied to each chain
MIN_BID        = 0.05    # discard contracts with bid < this
MAX_SPREAD_PCT = 0.25    # discard if (ask-bid)/mid > this
MIN_OPEN_INT   = 10      # discard contracts with open interest < this

# Expiry range
MIN_EXPIRY_DAYS = 7      # ignore very near-term contracts
MAX_EXPIRY_DAYS = 365    # ignore options expiring beyond 1 year

# Strike selection: N closest strikes below ATM + N closest at/above ATM = 2N total
N_STRIKES_EACH_SIDE = 50   # → up to 100 strikes total per expiry

# Strike spacing: sample every Nth available strike, anchored at ATM
# QQQ has $1 increments → STRIKE_STEP=2 gives $2 spacing, =3 gives $3 spacing
STRIKE_STEP = 2

# Skip weekly expirations — keep only standard monthly (3rd Friday) expiries
MONTHLY_ONLY = True

# ── Section 4 (Historical Backfill) ─────────────────────────────────────────
# Set True only when backfill is confirmed working (market hours, valid subscription)
BACKFILL_ENABLED = False

# How many calendar days back to backfill (Section 4)
BACKFILL_DAYS = 30

---
## 2. Connect to IB Gateway

In [ ]:
# ── 2. Connect ──────────────────────────────────────────────────────────────
ib = IB()
ib.connect(IB_HOST, IB_PORT, clientId=CLIENT_ID, readonly=True)
if not ib.isConnected():
    raise RuntimeError(f"Could not connect to IB Gateway at {IB_HOST}:{IB_PORT}. "
                       "Is IB Gateway running with API access enabled?")
print(f"Connected: {ib.isConnected()}  |  account: {ib.managedAccounts()}")

---
## 3. Live Option Chain Snapshot

In [ ]:
# ── Helper: fetch chain for one ticker ──────────────────────────────────────
def _is_monthly_expiry(exp_str: str) -> bool:
    """True if exp_str (YYYYMMDD) is the 3rd Friday of its month."""
    d = datetime.strptime(exp_str, "%Y%m%d").date()
    first = d.replace(day=1)
    days_to_first_friday = (4 - first.weekday()) % 7   # weekday 4 = Friday
    third_friday = first.replace(day=1 + days_to_first_friday + 14)
    return d == third_friday


def _get_spot(ib: IB, ticker: str) -> float:
    """Return last trade price for the underlying equity."""
    stk = Stock(ticker, "SMART", "USD")
    ib.qualifyContracts(stk)
    [td] = ib.reqTickers(stk)
    spot = td.last if td.last and td.last > 0 else td.close
    if not spot or spot <= 0:
        raise ValueError(f"Cannot get spot for {ticker}")
    return float(spot)


def _build_chain_df(
    ib: IB,
    ticker: str,
    spot: float,
    min_bid: float           = MIN_BID,
    max_spread_pct: float    = MAX_SPREAD_PCT,
    min_oi: int              = MIN_OPEN_INT,
    n_strikes_each_side: int = N_STRIKES_EACH_SIDE,
    strike_step: int         = STRIKE_STEP,
    min_expiry_days: int     = MIN_EXPIRY_DAYS,
    max_expiry_days: int     = MAX_EXPIRY_DAYS,
    monthly_only: bool       = MONTHLY_ONLY,
) -> pd.DataFrame:
    """
    Fetch a live option chain snapshot from IBKR for *ticker*.

    Keeps monthly (3rd-Friday) expiries only and selects the
    n_strikes_each_side closest strikes below and at/above spot,
    sampled every strike_step to widen the grid spacing.
    """
    # 1. Get contract parameters
    stk = Stock(ticker, "SMART", "USD")
    ib.qualifyContracts(stk)
    chains = ib.reqSecDefOptParams(ticker, "", "STK", stk.conId)
    if not chains:
        raise RuntimeError(f"No option chain found for {ticker}")

    print(f"  Chains found: {[(c.exchange, c.tradingClass, len(c.strikes)) for c in chains]}")

    # Prefer SMART exchange + exact trading class to avoid ambiguous contracts
    smart_match = [c for c in chains if c.exchange == "SMART" and c.tradingClass == ticker]
    any_match   = [c for c in chains if c.tradingClass == ticker]
    chain = (smart_match or any_match or [max(chains, key=lambda c: len(c.strikes))])[0]
    trading_class = chain.tradingClass
    print(f"  Using: exchange={chain.exchange}, tradingClass={trading_class}, strikes={len(chain.strikes)}")

    # 2. Filter expiries: date window + optional monthly-only
    today = pd.Timestamp.today().normalize()
    expiries = sorted(
        e for e in chain.expirations
        if min_expiry_days < (pd.to_datetime(e) - today).days <= max_expiry_days
        and (not monthly_only or _is_monthly_expiry(e))
    )
    print(f"  Expiries after filter ({'monthly only' if monthly_only else 'all'}): {len(expiries)}  {expiries}")

    # 3. Strike selection: sample every strike_step, anchored at ATM so near-the-money
    #    strikes are always included.  Collect up to n_strikes_each_side on each side.
    all_strikes = sorted(chain.strikes)
    below_raw   = [k for k in all_strikes if k <  spot]   # ascending; last = closest to ATM
    above_raw   = [k for k in all_strikes if k >= spot]   # ascending; first = closest to ATM

    # Reverse below list so index 0 is closest to ATM, then step outward
    below_atm = sorted(below_raw[::-1][::strike_step][:n_strikes_each_side])
    above_atm = above_raw[::strike_step][:n_strikes_each_side]
    strikes   = below_atm + above_atm

    effective_step = strike_step * (all_strikes[1] - all_strikes[0]) if len(all_strikes) > 1 else "?"
    print(f"  Strikes selected: {len(strikes)}  "
          f"({len(below_atm)} below + {len(above_atm)} at/above ATM={spot:.2f}, "
          f"step={strike_step} → ~${effective_step:.0f} spacing)")

    # 4. Build contract objects — specify tradingClass to avoid "Ambiguous contract" errors
    contracts = []
    for exp in expiries:
        for k in strikes:
            for right in ("C", "P"):
                contracts.append(Option(ticker, exp, k, right, "SMART", tradingClass=trading_class))

    print(f"  Total contracts to qualify: {len(contracts)}")

    # Qualify in batches of 200 — unknown contracts (strike not listed for that expiry) are dropped
    qualified = []
    for i in range(0, len(contracts), 200):
        qualified.extend(ib.qualifyContracts(*contracts[i:i+200]))
        time.sleep(0.2)

    print(f"  Qualified: {len(qualified)} contracts")

    if not qualified:
        raise RuntimeError(f"No qualified contracts for {ticker}. "
                           "Check IB Gateway connection.")

    # 5. Use frozen/delayed data so modelGreeks are populated outside market hours
    ib.reqMarketDataType(4)  # 1=live, 2=frozen, 3=delayed, 4=delayed-frozen

    CHUNK = 100
    tickers_data = []
    for i in range(0, len(qualified), CHUNK):
        batch = qualified[i:i+CHUNK]
        tds   = ib.reqTickers(*batch)
        tickers_data.extend(tds)
        time.sleep(1.0)

    # 6. Parse into rows
    rows = []
    for td in tickers_data:
        c   = td.contract
        bid = td.bid  if td.bid  and td.bid  > 0 else np.nan
        ask = td.ask  if td.ask  and td.ask  > 0 else np.nan
        mid = (bid + ask) / 2 if (not np.isnan(bid) and not np.isnan(ask)) else np.nan

        iv = delta = gamma = vega = theta = np.nan
        if td.modelGreeks:
            g = td.modelGreeks
            iv    = g.impliedVol if g.impliedVol and g.impliedVol > 0 else np.nan
            delta = g.delta      if g.delta  is not None else np.nan
            gamma = g.gamma      if g.gamma  is not None else np.nan
            vega  = g.vega       if g.vega   is not None else np.nan
            theta = g.theta      if g.theta  is not None else np.nan

        oi = td.callOpenInterest if c.right == "C" else td.putOpenInterest
        if oi is None:
            oi = np.nan

        rows.append({
            "strike":           float(c.strike),
            "expiry":           c.lastTradeDateOrContractMonth,
            "option_type":      "call" if c.right == "C" else "put",
            "bid":              bid,
            "ask":              ask,
            "option_price":     mid,
            "implied_vol":      iv,
            "delta":            delta,
            "gamma":            gamma,
            "vega":             vega,
            "theta":            theta,
            "open_interest":    oi,
            "underlying_price": spot,
        })

    df = pd.DataFrame(rows)
    print(f"  Rows before filters: {len(df)}  (non-null IV: {df['implied_vol'].notna().sum()}, "
          f"non-null bid: {df['bid'].notna().sum()})")

    # 7. Liquidity filters
    # implied_vol is always required; bid/ask/OI filters only apply where the field is populated
    # (frozen/delayed data often has Greeks but no bid/ask)
    df = df.dropna(subset=["implied_vol"])

    has_bid = df["bid"].notna()
    df = df[~has_bid | (df["bid"] >= min_bid)]

    has_both = df["bid"].notna() & df["ask"].notna()
    spread_pct = (df["ask"] - df["bid"]) / df["option_price"].clip(lower=1e-6)
    df = df[~has_both | (spread_pct <= max_spread_pct)]

    if min_oi > 0:
        has_oi = df["open_interest"].notna()
        df = df[~has_oi | (df["open_interest"] >= min_oi)]

    df["expiry"] = pd.to_datetime(df["expiry"], format="%Y%m%d").dt.date.astype(str)
    return df.reset_index(drop=True)

In [ ]:
# ── 3. Fetch live chain and save ─────────────────────────────────────────────
today_str = date.today().isoformat()

for tkr in TICKERS:
    print(f"\n{'='*60}")
    print(f"Fetching {tkr} …")
    try:
        spot  = _get_spot(ib, tkr)
        print(f"  spot = {spot:.2f}")
        chain = _build_chain_df(ib, tkr, spot)
        print(f"  raw rows after filters: {len(chain)}")

        n = save_option_snapshot(chain, tkr, db_path=DB_PATH, source="ibkr")
        print(f"  saved {n} new rows  (snapshot_date={today_str})")

    except Exception as e:
        print(f"  ERROR: {e}")

print("\nDone. DB summary:")
display(list_snapshots(db_path=DB_PATH))

---
## 4. Historical IV Backfill

IBKR `reqHistoricalData` with `whatToShow='OPTION_IMPLIED_VOLATILITY'` returns a
daily bar series (open/high/low/close) of the IV for a specific option contract.

Strategy here:
1. Get today's live chain to know which contracts exist
2. For each contract, request N days of IV history
3. For each historical date, construct a synthetic row and call `save_option_snapshot`
   with `snapshot_date=` override

> **Rate limits**: IBKR allows 50 historical data requests per 10 seconds.
> We enforce a `HIST_DELAY` between requests. For large chains (1000+ contracts)
> this can take several minutes.

In [ ]:
# ── 4a. Config for backfill ──────────────────────────────────────────────────
BACKFILL_TICKER = "QQQ"   # ticker to backfill (one at a time to control rate)
BACKFILL_DAYS   = 30       # calendar days back from today
HIST_DELAY      = 0.22     # seconds between historical requests (≤50/10 s limit)

In [ ]:
# ── 4b. Build list of contracts to backfill ──────────────────────────────────
if not BACKFILL_ENABLED:
    raise RuntimeError("Backfill disabled — set BACKFILL_ENABLED=True in the config cell to run")

print(f"Getting contract list for {BACKFILL_TICKER} …")
bf_spot  = _get_spot(ib, BACKFILL_TICKER)
bf_chain = _build_chain_df(
    ib, BACKFILL_TICKER, bf_spot,
    min_bid=0.0, max_spread_pct=1.0, min_oi=0,   # no filters — just need contract list
)
print(f"  {len(bf_chain)} contracts to backfill")
bf_chain.head(3)

In [ ]:
# ── 4c. Fetch historical IV per contract ─────────────────────────────────────
if not BACKFILL_ENABLED:
    raise RuntimeError("Backfill disabled — set BACKFILL_ENABLED=True in the config cell to run")

from scipy.optimize import brentq
from skew.utils import bs_price_forward

# Resolve trading class
_stk = Stock(BACKFILL_TICKER, "SMART", "USD")
ib.qualifyContracts(_stk)
_chains      = ib.reqSecDefOptParams(BACKFILL_TICKER, "", "STK", _stk.conId)
_smart_match = [c for c in _chains if c.exchange == "SMART" and c.tradingClass == BACKFILL_TICKER]
_any_match   = [c for c in _chains if c.tradingClass == BACKFILL_TICKER]
bf_trading_class = (_smart_match or _any_match or [_chains[0]])[0].tradingClass
print(f"Trading class: {bf_trading_class}")

# ── Probe: check which data type returns bars (paper vs live account) ────────
_test_row   = bf_chain.iloc[0]
_test_con   = Option(BACKFILL_TICKER, _test_row["expiry"].replace("-", ""),
                     _test_row["strike"],
                     "C" if _test_row["option_type"] == "call" else "P",
                     "SMART", tradingClass=bf_trading_class)
ib.qualifyContracts(_test_con)

_probe_iv  = ib.reqHistoricalData(_test_con, endDateTime="", durationStr="3 D",
                                   barSizeSetting="1 day",
                                   whatToShow="OPTION_IMPLIED_VOLATILITY",
                                   useRTH=False, formatDate=1)
_probe_mid = ib.reqHistoricalData(_test_con, endDateTime="", durationStr="3 D",
                                   barSizeSetting="1 day", whatToShow="MIDPOINT",
                                   useRTH=False, formatDate=1)

USE_IV_BARS = bool(_probe_iv)
print(f"OPTION_IMPLIED_VOLATILITY: {len(_probe_iv)} bars  |  MIDPOINT: {len(_probe_mid)} bars")

if not _probe_iv and not _probe_mid:
    raise RuntimeError(
        "No historical data returned for either OPTION_IMPLIED_VOLATILITY or MIDPOINT.\n"
        "Check IBKR market data subscriptions, or run daily live snapshots (Section 3) instead."
    )

method = "OPTION_IMPLIED_VOLATILITY" if USE_IV_BARS else "MIDPOINT + Black-Scholes IV"
print(f"Method: {method}\n")

# ── If using MIDPOINT, fetch historical spot prices once for all dates ───────
spot_by_date: dict[str, float] = {}
if not USE_IV_BARS:
    _spot_bars = ib.reqHistoricalData(
        _stk, endDateTime="", durationStr=f"{BACKFILL_DAYS} D",
        barSizeSetting="1 day", whatToShow="TRADES",
        useRTH=False, formatDate=1,
    )
    spot_by_date = {str(b.date)[:10]: b.close for b in _spot_bars if b.close > 0}
    print(f"Spot history: {len(spot_by_date)} days — {sorted(spot_by_date.keys())}")

    def _bs_iv(price: float, is_call: bool, S: float, K: float, T: float,
               r: float = 0.0525, q: float = 0.0) -> float:
        """Black-Scholes IV inversion via Brentq. Returns NaN if impossible."""
        if T <= 1e-4 or price <= 0 or S <= 0:
            return np.nan
        D  = np.exp(-r * T)
        F  = S * np.exp((r - q) * T)
        intrinsic = max(D * (F - K), 0.0) if is_call else max(D * (K - F), 0.0)
        if price <= intrinsic + 1e-8:
            return np.nan
        try:
            return brentq(
                lambda v: bs_price_forward(is_call, F, K, v, T, D) - price,
                1e-4, 20.0, xtol=1e-5, maxiter=100,
            )
        except Exception:
            return np.nan

# ── Main backfill loop ───────────────────────────────────────────────────────
duration_str = f"{BACKFILL_DAYS} D"
hist_rows: dict[str, list] = {}
errors  = 0
skipped = 0

for i, row in bf_chain.iterrows():
    exp_ibkr = row["expiry"].replace("-", "")
    right    = "C" if row["option_type"] == "call" else "P"
    is_call  = (right == "C")
    K        = row["strike"]
    exp_date = pd.to_datetime(row["expiry"]).date()

    opt_con   = Option(BACKFILL_TICKER, exp_ibkr, K, right, "SMART",
                       tradingClass=bf_trading_class)
    qualified = ib.qualifyContracts(opt_con)
    if not qualified or not qualified[0].conId:
        skipped += 1
        continue

    try:
        bars = ib.reqHistoricalData(
            qualified[0], endDateTime="", durationStr=duration_str,
            barSizeSetting="1 day",
            whatToShow="OPTION_IMPLIED_VOLATILITY" if USE_IV_BARS else "MIDPOINT",
            useRTH=False, formatDate=1,
        )
    except Exception:
        errors += 1
        time.sleep(HIST_DELAY)
        continue

    for bar in bars:
        d_str = str(bar.date)[:10]

        if USE_IV_BARS:
            iv = bar.close if bar.close > 0 else np.nan
        else:
            S = spot_by_date.get(d_str)
            if S is None:
                continue
            T  = max((exp_date - datetime.strptime(d_str, "%Y-%m-%d").date()).days / 365.0, 1e-4)
            iv = _bs_iv(bar.close, is_call, S, K, T)

        if np.isnan(iv):
            continue

        hist_rows.setdefault(d_str, []).append({
            "strike":           K,
            "expiry":           row["expiry"],
            "option_type":      row["option_type"],
            "implied_vol":      iv,
            "underlying_price": spot_by_date.get(d_str, bf_spot),
            "bid":              np.nan,
            "ask":              np.nan,
            "option_price":     bar.close if not USE_IV_BARS else np.nan,
        })

    time.sleep(HIST_DELAY)
    if (i + 1) % 50 == 0:
        total_rows = sum(len(v) for v in hist_rows.values())
        print(f"  {i+1}/{len(bf_chain)} done  "
              f"({errors} errors, {skipped} skipped, {total_rows} rows across {len(hist_rows)} dates)")

total_rows = sum(len(v) for v in hist_rows.values())
print(f"\nFinished: {len(hist_rows)} dates, {total_rows} rows, {errors} errors, {skipped} skipped")
print("Dates:", sorted(hist_rows.keys()))

In [ ]:
# ── 4d. Save historical rows to DB ───────────────────────────────────────────
if not BACKFILL_ENABLED:
    raise RuntimeError("Backfill disabled — set BACKFILL_ENABLED=True in the config cell to run")

total_saved = 0
for d_str, rows in sorted(hist_rows.items()):
    df_day = pd.DataFrame(rows)
    n = save_option_snapshot(
        df_day,
        BACKFILL_TICKER,
        db_path=DB_PATH,
        source="ibkr_hist",
        snapshot_date=d_str,   # override date for historical rows
    )
    total_saved += n
    print(f"  {d_str}: {len(df_day)} rows → {n} new")

print(f"\nTotal new rows saved: {total_saved}")
print("DB summary:")
display(list_snapshots(db_path=DB_PATH))

---
## 5. Disconnect

In [ ]:
ib.disconnect()
print("Disconnected.")

---
## Appendix: Column mapping reference

| IBKR field | DB column |
|---|---|
| `modelGreeks.impliedVol` | `implied_vol` |
| `modelGreeks.delta` | `delta` |
| `modelGreeks.gamma` | `gamma` |
| `modelGreeks.vega` | `vega` |
| `modelGreeks.theta` | `theta` |
| `bid` | `bid` |
| `ask` | `ask` |
| `(bid+ask)/2` | `option_price` |
| `contract.strike` | `strike` |
| `contract.lastTradeDateOrContractMonth` → YYYY-MM-DD | `expiry` |
| `"C"→"call", "P"→"put"` | `option_type` |
| `callOpenInterest` / `putOpenInterest` | `open_interest` |

Historical IV bars (`whatToShow='OPTION_IMPLIED_VOLATILITY'`) return only the IV series.
Bid/ask/price are stored as NaN for historical rows — they are not needed for skew analytics
(`pdf_calc.ipynb` uses `implied_vol` directly).